## Строим паплайн обучения моделей для классификации 

Здесь предобработка исходных данных текстовых полей после кодирования в эмбеддинги предполагает выделение 3PCA из эмбеддинга + его усреднение по всем текстовым полям - purpose, articles__name, projects__name, counterparties__name.

In [1]:
import pandas as pd
import numpy as np
import random
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

# ⏳ прогресс-бары
import tqdm as tqdm_lib
from tqdm import tqdm

# 🧠 обработка текста и NLP
import spacy
try:
    import transformers
    from transformers import AutoModel, AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"], stdout=subprocess.DEVNULL)
    import transformers
    from transformers import AutoModel, AutoTokenizer

try:
    import nlpaug
    import nlpaug.augmenter.char as nac
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nlpaug"], stdout=subprocess.DEVNULL)
    import nlpaug
    import nlpaug.augmenter.char as nac
    
    
# 🤖 pyTorch
import torch

# загрузка моделей и функци для предобработки данных
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import label_binarize

from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

import catboost
from catboost import CatBoostClassifier, utils


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

2025-12-20 14:26:35.963215: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-20 14:26:37.718403: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-20 14:26:41.588951: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [3]:
# создаем функцию очистки текста
def _clean_single_text(text):
    return re.sub(r"[^\w\s]", " ", text.lower())

# создаем функцию предобработки текста
def preprocess_texts_optimized(texts, nlp_model_name,
                               batch_size_cpu=256,
                               num_processes_for_cleaning=-1,
                               num_processes_for_spacy_cpu=-1):
    
    print(f"🔍 Запуск предобработки для {len(texts)} текстов...")
    
    # предварительная очистка текстов
    cleaned_texts = [_clean_single_text(text) for text in tqdm(texts, desc="Очистка")]

    # spaCy и лемматизация
    nlp = None
    processed_lemmas = []
    
    # загрузка NLP модели
    print(f"🔍 Загрузка spaCy модели: '{nlp_model_name}'.")
    nlp = spacy.load(nlp_model_name)

    # используем n_process для параллелизации
    if num_processes_for_spacy_cpu == -1:
        cpu_count = os.cpu_count()
        num_processes_for_spacy_cpu = max(1, cpu_count - 1)
    
    print(f"🔍 Лемматизация будет выполнена в {num_processes_for_spacy_cpu} потоках.")
    
    for doc in tqdm(nlp.pipe(cleaned_texts, batch_size=batch_size_cpu, n_process=num_processes_for_spacy_cpu), total=len(cleaned_texts), desc="Лемматизация (CPU)"):
        lemmas = [token.lemma_ for token in doc]
        processed_lemmas.append(" ".join(lemmas))
    
    print(f"🔍 Предобработка завершена. Обработано {len(processed_lemmas)} текстов.")
    return processed_lemmas

# создаем функцию для получения усредненного эмбеддинга текста
def get_embeddings_batch(texts, tokenizer, model, device, batch_size=64):
    texts = list(texts)
    embeddings = []

    print(f"🔍 Начало генерации эмбеддингов для {len(texts)} текстов на устройстве '{device}'.")
    for i in tqdm(range(0, len(texts), batch_size), desc="Генерация эмбеддингов"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        # Переносим каждый тензор на устройство
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # берем attention mask (1 — реальные токены, 0 — паддинг)
        mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size())
        masked_embeddings = outputs.last_hidden_state * mask

        # считаем среднее только по непаддинговым токенам
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts

        embeddings.extend(mean_pooled.cpu().numpy())
    
    print(f"🔍 Генерация эмбеддингов завершена")
    
    return embeddings

# функция предобработки входного датасета
def prepare_data(data, is_train, scaler=None, ipca=None, 
                                 scaler_art=None, ipca_art=None,
                                 scaler_pro=None, ipca_pro=None,
                                 scaler_cou=None, ipca_cou=None):
    
    data = data.drop(
            [
                "robot_id",
                "accounts__id",
                "articles__id",
                "articles__user_id",
                "projects__id",
                "projects__user_id",
                "counterparties__id",
                "counterparties__user_id",
                "robots__user_id",
                "article_alternative_names__user_id",
            ],
            axis=1,
        )

    # поправим типы данных и заполним пропуски метками missing (для текстовых значений категорий) и 0 для пропущенных ID
    data[
        [
            "articles__parent_id",
            "projects__parent_id",
            "counterparties__parent_id",
            "robots__id",
        ]
    ] = (
        data[
            [
                "articles__parent_id",
                "projects__parent_id",
                "counterparties__parent_id",
                "robots__id",
            ]
        ]
        .fillna(0)
        .astype("int64")
    )

    data["purpose"] = data["purpose"].fillna("missing")
    data["articles__name"] = data["articles__name"].fillna("missing")
    data["projects__name"] = data["projects__name"].fillna("missing")
    data["counterparties__name"] = data["counterparties__name"].fillna("missing")

    # конвертируем дату в datetime
    data["date"] = pd.to_datetime(data["date"], yearfirst=True, errors='coerce')

    # и убираем записи из будущего и меньше нуля (и такое бывает)
    yesterday = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    data = data[data["date"] <= yesterday]
    #data = data[data["payments_amount"] > 0]

    # переименуем и поправим тип столбца с фондами
    data = data.rename(columns={"accounts__user_id": "user_id"})
    data["user_id"] = data["user_id"].fillna(0).astype("int64")

    # кодируем текстовые поля
    # сначала очищаем и лемматизируем тексты
    data["clean_purpose"] = preprocess_texts_optimized(texts=data["purpose"],nlp_model_name="ru_core_news_sm")

    # грузим модели 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
    model = AutoModel.from_pretrained("DeepPavlov/rubert-base-cased")
    model = model.to(device)
    model.eval()
    
    # и запускаем генерацию эмбеддингов в назначении платежа
    data["purpose_emb"] = get_embeddings_batch(data["clean_purpose"], tokenizer, model, device)

    # 1. усредняем эмбеддинг в одно число
    data["purpose_mean"] = data["purpose_emb"].apply(lambda x: float(np.mean(x)))

    # 2. выделяем три главные компоненты с предварительным масштабированием и по батчам  
    batch_size = 10_000
    
    if is_train:
        scaler = StandardScaler()
        ipca = IncrementalPCA(n_components=3)

        # обучаем скейлер
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            scaler.partial_fit(batch)

        # обучаем PCA на масштабированных данных
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler.transform(batch)
            ipca.partial_fit(batch_scaled)

    # применяем трансформацию ко всему массиву
    transformed_batches = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к эмбеддингам"):
        batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler.transform(batch)
        transformed_batches.append(ipca.transform(batch_scaled))
        
    purpose_pca_features = np.vstack(transformed_batches)

    # делаем датафрейм
    pca_column_names = [f"purpose_pca_{i+1}" for i in range(3)]
    data[pca_column_names] = purpose_pca_features

    # удалим ненужные столбцы
    data.drop(columns=["purpose", "clean_purpose", "purpose_emb"], inplace=True)

    # генерируем эмбеддинги для названий статей
    # сначала очищаем и лемматизируем тексты
    data["clean_articles__name"] = preprocess_texts_optimized(texts=data["articles__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["articles__name_emb"] = get_embeddings_batch(data["clean_articles__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["articles__name_mean"] = data["articles__name_emb"].apply(lambda x: float(np.mean(x)))
    
    # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_art = StandardScaler()
        ipca_art = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (articles)"):
            batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size])
            scaler_art.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (articles)"):
            batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_art.transform(batch)
            ipca_art.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_art = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к articles embeddings"):
        batch = np.vstack(data["articles__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_art.transform(batch)
        transformed_batches_art.append(ipca_art.transform(batch_scaled))

    articles_pca_features = np.vstack(transformed_batches_art)

    # 4) создаём колонки
    art_pca_colnames = [f"articles_pca_{i+1}" for i in range(3)]
    data[art_pca_colnames] = articles_pca_features
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["articles__name", "clean_articles__name", "articles__name_emb"], inplace=True)

    # генерируем эмбеддинги для названий проектов
    # сначала очищаем и лемматизируем тексты
    data["clean_projects__name"] = preprocess_texts_optimized(texts=data["projects__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["projects__name_emb"] = get_embeddings_batch(data["clean_projects__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["projects__name_mean"] = data["projects__name_emb"].apply(lambda x: float(np.mean(x)))
    
    
     # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_pro = StandardScaler()
        ipca_pro = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (projects)"):
            batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size])
            scaler_pro.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (projects)"):
            batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_pro.transform(batch)
            ipca_pro.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_pro = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к projects embeddings"):
        batch = np.vstack(data["projects__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_pro.transform(batch)
        transformed_batches_pro.append(ipca_pro.transform(batch_scaled))

    projects_pca_features = np.vstack(transformed_batches_pro)

    # 4) создаём колонки
    pro_pca_colnames = [f"projects_pca_{i+1}" for i in range(3)]
    data[pro_pca_colnames] = projects_pca_features
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["projects__name", "clean_projects__name", "projects__name_emb"], inplace=True)
    
    # генерируем эмбеддинги для названий доноров
    # сначала очищаем и лемматизируем тексты
    data["clean_counterparties__name"] = preprocess_texts_optimized(texts=data["counterparties__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["counterparties__name_emb"] = get_embeddings_batch(data["clean_counterparties__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["counterparties__name_mean"] = data["counterparties__name_emb"].apply(lambda x: float(np.mean(x)))
    
    
     # ----------------------------------------------------------
    #  Выделяем PCA (тоже 3 компоненты), полностью аналогично purpose
    # ----------------------------------------------------------

    batch_size = 10_000

    if is_train:
        scaler_cou = StandardScaler()
        ipca_cou = IncrementalPCA(n_components=3)

        # 1) обучаем scaler
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler (counterparties)"):
            batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size])
            scaler_cou.partial_fit(batch)

        # 2) обучаем PCA
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA (counterparties)"):
            batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler_cou.transform(batch)
            ipca_cou.partial_fit(batch_scaled)

    # 3) применяем трансформацию ко всем данным
    transformed_batches_cou = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к counterparties embeddings"):
        batch = np.vstack(data["counterparties__name_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler_cou.transform(batch)
        transformed_batches_cou.append(ipca_cou.transform(batch_scaled))

    counterparties_pca_features = np.vstack(transformed_batches_cou)

    # 4) создаём колонки
    cou_pca_colnames = [f"counterparties_pca_{i+1}" for i in range(3)]
    data[cou_pca_colnames] = counterparties_pca_features
    
    
    
    
    
    # удалим ненужные столбцы
    data.drop(columns=["counterparties__name", "clean_counterparties__name", "counterparties__name_emb"], inplace=True)

    
    # кодируем таргет словарем кодиррования uc
    uc_codex = {"пожертвования от физических лиц (напрямую)":1,
            "пожертвования через платформы":2,
            "пожертвования от юридических лиц (напрямую)":3,
            "прочие недоходные операции":4,
            "продажа услуг":5,
            "продажа товаров":6,
            "финансовые доходы":7,
            "членские и учредительские взносы":8,
            "гранты субсидии конкурсы":9}

    data['target'] = data['universal_category'].map(uc_codex)
    
    # добавляем признаки, производные из даты с учетом анализа в EDA
    #data["day_of_week"] = data["date"].dt.weekday
    #data["day_of_week_sin"] = np.sin(2 * np.pi * data["day_of_week"] / 7).round(6)
    #data["month"] = data["date"].dt.month
    #data["is_weekend"] = (data["day_of_week"] >= 5).astype(int)


    # сбрасываем неактуальные столбцы
    data = data.drop(columns=[ 
        'date', 'expenditure','universal_category',
        'payments_amount','user_id','account_id', 
        'contractor_id', 'article_id', 'project_id', 
        'counterpartie_id', 'donor_id', 'donor_cat_id',
        'articles__parent_id', 'projects__parent_id',
        'counterparties__parent_id', 'robots__id'])

    return data,scaler,ipca,scaler_art,ipca_art,scaler_pro,ipca_pro,scaler_cou,ipca_cou


### Загрузим, разделим и подготовим данные для обучения моделей

In [4]:
# выделим тестовую выборку соответствующую проверкам LLM
X_test_primer = pd.read_csv('X_test_balanced_id.csv', index_col=0)
data = pd.read_parquet("data_download_09102025_uc.parquet")

data_test = data[data['id'].isin(X_test_primer['id'])]
order = X_test_primer["id"]
data_test = data_test.set_index("id").loc[order].reset_index()
data = data[~data['id'].isin(X_test_primer['id'])]

data_test.to_csv('data_test.csv', index=False)
data.to_parquet('data.parquet', index=False)

In [5]:
# отбираем строки, где нет пропусков в целевом признаке 
data.dropna(subset=['universal_category'], inplace=True)
#data.drop(columns=['id'], inplace=True)

train_df, val_df = train_test_split(data, test_size=0.1, random_state=RANDOM_STATE, stratify=data['universal_category'])

train_df.shape, val_df.shape

((518305, 31), (57590, 31))

In [6]:
# добавим шума в тренировочные данные

import random
import nlpaug.augmenter.char as nac


delete_aug = nac.RandomCharAug(action="delete", aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)
insert_aug = nac.RandomCharAug(action="insert", aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)
swap_aug   = nac.RandomCharAug(action="swap",   aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)
sub_aug    = nac.RandomCharAug(action="substitute", aug_char_min=1, aug_char_max=1, aug_char_p=0.01, aug_word_p=0.15)

augs = [delete_aug, insert_aug, swap_aug, sub_aug]

def corrupt_text(text, n=1):
    if not isinstance(text, str):
        return text
    
    out = text
    for _ in range(n):
        aug = random.choice(augs)
        tmp = aug.augment(out)

        if isinstance(tmp, list):
            tmp = tmp[0]

        out = tmp
    return out


# 1. Отбор 20% строк для всех классов на трейне
df_aug_list = []

for cls in train_df["universal_category"].unique():
    group = train_df[train_df["universal_category"] == cls]
    
    sample = group.sample(
        frac=0.20,
        random_state=RANDOM_STATE
    )
    
    df_aug_list.append(sample)

df_aug = pd.concat(df_aug_list, ignore_index=True)

display(df_aug.shape)
display(df_aug["universal_category"].value_counts())


# 2. Аугментация текстовых столбцов
df_aug['purpose']               = df_aug['purpose'].astype(str).apply(lambda x: corrupt_text(x, n=2))
df_aug['articles__name']        = df_aug['articles__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))
df_aug['projects__name']        = df_aug['projects__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))
df_aug['counterparties__name']  = df_aug['counterparties__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))


# 3. Добавляем шумные данные в train
train_df = pd.concat([train_df, df_aug], ignore_index=True)

display(train_df.shape, train_df.tail())


# 4. Отбор 20% строк для всех классов на валидации
df_aug_list = []

for cls in val_df["universal_category"].unique():
    group = val_df[val_df["universal_category"] == cls]
    
    sample = group.sample(
        frac=0.20,
        random_state=RANDOM_STATE
    )
    
    df_aug_list.append(sample)

df_aug = pd.concat(df_aug_list, ignore_index=True)

display(df_aug.shape)
display(df_aug["universal_category"].value_counts())


# 5. Аугментация текстовых столбцов
df_aug['purpose']               = df_aug['purpose'].astype(str).apply(lambda x: corrupt_text(x, n=2))
df_aug['articles__name']        = df_aug['articles__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))
df_aug['projects__name']        = df_aug['projects__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))
df_aug['counterparties__name']  = df_aug['counterparties__name'].astype(str).apply(lambda x: corrupt_text(x, n=1))


# 3. Добавляем шумные данные в train
val_df = pd.concat([val_df, df_aug], ignore_index=True)

display(val_df.shape,val_df.tail())


(103661, 31)

пожертвования от физических лиц (напрямую)     60845
пожертвования через платформы                  34341
пожертвования от юридических лиц (напрямую)     3053
прочие недоходные операции                      1963
продажа услуг                                   1660
продажа товаров                                 1172
финансовые доходы                                405
членские и учредительские взносы                 150
гранты субсидии конкурсы                          72
Name: universal_category, dtype: int64

(621966, 31)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,universal_category
621961,386608,629,1126,2024-09-18,1430552.00,ПеречисNление срдесVтв (2) по договору о рпедосlтавлении рганта Президента Российскfой Федерации на развитие Гnражда...,25890,incoming,1579,5558,1570,1057,0,629.0,712.0,25890.0,712.0,25888.0,Гратн,1579.0,712.0,0.0,Ты в прядке,NaN,NaN,NaN,Non,1057.0,712.0,NaN,гранты субсидии конкурсы
621962,649788,1391,44418,2024-12-25,368.00,( 609. 0b70(. 0120P6l1320. 635. 24B 021331D0655^) Л / С 0Vk609171371 БО 1o131713724000519 Суб. в цел. фин. Rоpесп. м...,35024,incoming,2514,0,0,-1,7795,1391.0,202.0,35024.0,202.0,35022.0,Субсидии !ФДО,2514.0,202.0,2511.0,jПрочие МО,7795.0,202.0,2077.0,Yуниципалитеты,NaN,NaN,NaN,гранты субсидии конкурсы
621963,1067920,2166,1126,2025-07-18,2572035.50,Перечисление $ uредств по договору о предоставлении гра2нта Президента Российской 5едерации на развитие гражданского...,37051,incoming,3008,0,0,5845,8407,2166.0,877.0,37051.0,877.0,37050.0,Грнаты,3008.0,877.0,0.0,УставMная деятельность,8407.0,877.0,8398.0,ФПГ,5845.0,877.0,NaN,гранты субсидии конкурсы
621964,472988,1977,223,2024-08-28,3314894.40,Епречисление по дог. № ИС - 27. 12 / 2032 от 27. 12. 2023 год о предосатвлении целвеого гатна в рамакх релаиации бла...,39890,incoming,3261,0,0,-1,8575,1977.0,891.0,39890.0,891.0,37785.0,П7ожертвования от грантодателей,3261.0,891.0,0.0,Жизнь в общесвте,8575.0,891.0,8568.0,"_СберВместе "" Снова вместе """,NaN,NaN,NaN,гранты субсидии конкурсы
621965,316103,1391,44417,2024-05-07,224513.16,"( 923. 703. 013167700. 635. 24B л / с 0233025990) л / с 0392327351 БО 0207305202400095 Суб. на ине цеил, оеспеч. обе...",35024,incoming,2512,0,0,-1,7795,1391.0,202.0,35024.0,202.0,35022.0,Субсидии ПФqО,2512.0,202.0,2511.0,"МО "" Гоuод Сарапул """,7795.0,202.0,2077.0,М%ниципалитеты,NaN,NaN,NaN,гранты субсидии конкурсы


(11518, 31)

пожертвования от физических лиц (напрямую)     6761
пожертвования через платформы                  3816
пожертвования от юридических лиц (напрямую)     339
прочие недоходные операции                      218
продажа услуг                                   184
продажа товаров                                 130
финансовые доходы                                45
членские и учредительские взносы                 17
гранты субсидии конкурсы                          8
Name: universal_category, dtype: int64

(69108, 31)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,universal_category
69103,826659,2979,96992,2024-03-15,952958.94,Перечисление средсcв для открытия депDозита по депозитнVой сде% лке № UNV - 00000r1236405 от 15. 03. 2^24 сnогласно ...,27271,incoming,3256,0,0,-1,8065,2979.0,731.0,27271.0,731.0,27269.0,Гран,3256.0,731.0,3253.0,игре,8065.0,731.0,5876.0,средие,NaN,NaN,NaN,гранты субсидии конкурсы
69104,133537,112,1126,2023-11-29,2066621.94,Епречислене сердст (2) по договор о предосталвении гранта Президента Российской Федеарции на развитие граждансого об...,8179,incoming,533,0,0,-1,5820,112.0,185.0,8179.0,185.0,26229.0,Гран(,533.0,185.0,531.0,Хотим бMть вместе: подростки. От 8еории к практике инклюзии для деaей и подростков Санкт - Петербурга,5820.0,185.0,1905.0,ФПГ,NaN,NaN,NaN,гранты субсидии конкурсы
69105,982003,2565,122,2025-07-30,1596.00,"Возврат денежsых срWдств за закBз № 01673F784 (- g047, НДС 20% 266, 00 руб.",39767,incoming,2099,0,0,-1,9696,2565.0,782.0,39767.0,782.0,31791.0,ФПГ_202M5,2099.0,782.0,0.0,Ы_Возвратeы,9696.0,782.0,7035.0,ФОНД ПРЗИДЕНТСКИХ ГРАНТОВ,NaN,NaN,NaN,гранты субсидии конкурсы
69106,343842,211,29298,2024-03-27,125000.00,3Благотворительне пожерhтвование по дог. № 108t1778 от 27. 09. 203. Без НДС,35518,incoming,764,0,0,-1,7923,211.0,197.0,35518.0,197.0,35510.0,Грант Помщь рядом,764.0,197.0,0.0,Торческие проекты,7923.0,197.0,2048.0,Помощ7 рядом_Яндекс,NaN,NaN,NaN,гранты субсидии конкурсы
69107,801569,264,1126,2025-04-01,1002883.00,Пе2речисление сре8tдств по доnговору о предоставSлении граtнта Президента Российской Федерации на развитие граждаSнс...,7859,incoming,3895,0,0,-1,10135,264.0,176.0,7859.0,176.0,7857.0,Гант,3895.0,176.0,514.0,К устойчивому будущеcу,10135.0,176.0,1809.0,ФПГ - 20)5,NaN,NaN,NaN,гранты субсидии конкурсы


In [7]:
# загрузим подготовленный датасет с эмбеддингами, если его нет, то делаем полную подготовку данных с кодированием текстов

data_train_prepared_path = 'data_train_prepared.parquet'
scaler_emb_path = 'scaler.pkl'
ipca_emb_path = 'ipca.pkl'
scaler_art_emb_path = 'scaler_art.pkl'
ipca_art_emb_path = 'ipca_art.pkl'
scaler_pro_emb_path = 'scaler_pro.pkl'
ipca_pro_emb_path = 'ipca_pro.pkl'
scaler_cou_emb_path = 'scaler_cou.pkl'
ipca_cou_emb_path = 'ipca_cou.pkl'


if os.path.exists(data_train_prepared_path) and os.path.exists(scaler_emb_path) and os.path.exists(ipca_emb_path):
    data_train_prepared = pd.read_parquet(data_train_prepared_path)
    scaler = joblib.load(scaler_emb_path)
    ipca = joblib.load(ipca_emb_path)
    scaler_art = joblib.load(scaler_art_emb_path)
    ipca_art = joblib.load(ipca_art_emb_path)
    scaler_pro = joblib.load(scaler_pro_emb_path)
    ipca_pro = joblib.load(ipca_pro_emb_path)
    scaler_cou = joblib.load(scaler_cou_emb_path)
    ipca_cou = joblib.load(ipca_cou_emb_path)

else:
    data_train_prepared,scaler,ipca,scaler_art,ipca_art,scaler_pro,ipca_pro,scaler_cou,ipca_cou = prepare_data(train_df,is_train=True,scaler=None,ipca=None,
                                                                          scaler_art=None, ipca_art=None,
                                                                          scaler_pro=None, ipca_pro=None,
                                                                          scaler_cou=None, ipca_cou=None)

    data_train_prepared.to_parquet(data_train_prepared_path)
    joblib.dump(scaler, scaler_emb_path)
    joblib.dump(ipca, ipca_emb_path)
    joblib.dump(scaler_art, scaler_art_emb_path)
    joblib.dump(ipca_art, ipca_art_emb_path)
    joblib.dump(scaler_pro, scaler_pro_emb_path)
    joblib.dump(ipca_pro, ipca_pro_emb_path)
    joblib.dump(scaler_cou, scaler_cou_emb_path)
    joblib.dump(ipca_cou, ipca_cou_emb_path)

In [8]:
# предобрабатываем валидационный набор

data_val_prepared_path = 'data_val_prepared.parquet'

if os.path.exists(data_val_prepared_path):
    data_val_prepared = pd.read_parquet(data_val_prepared_path)
    
else:
    
    scaler_emb_path = 'scaler.pkl'
    ipca_emb_path = 'ipca.pkl'
    scaler_art_emb_path = 'scaler_art.pkl'
    ipca_art_emb_path = 'ipca_art.pkl'
    scaler_pro_emb_path = 'scaler_pro.pkl'
    ipca_pro_emb_path = 'ipca_pro.pkl'
    scaler_cou_emb_path = 'scaler_cou.pkl'
    ipca_cou_emb_path = 'ipca_cou.pkl'
    
    
    scaler = joblib.load(scaler_emb_path)
    ipca = joblib.load(ipca_emb_path)
    scaler_art = joblib.load(scaler_art_emb_path)
    ipca_art = joblib.load(ipca_art_emb_path)
    scaler_pro = joblib.load(scaler_pro_emb_path)
    ipca_pro = joblib.load(ipca_pro_emb_path)
    scaler_cou = joblib.load(scaler_cou_emb_path)
    ipca_cou = joblib.load(ipca_cou_emb_path)
    
    data_val_prepared,_,_,_,_,_,_,_,_ = prepare_data(val_df, is_train=False, scaler=scaler, ipca=ipca,
                                                                   scaler_art=scaler_art, ipca_art=ipca_art,
                                                                   scaler_pro=scaler_pro, ipca_pro=ipca_pro,
                                                                   scaler_cou=scaler_cou, ipca_cou=ipca_cou)

    data_val_prepared.to_parquet(data_val_prepared_path)


### Строим модель для обучения классификатора

In [9]:
# разделим признаки
X_train = data_train_prepared.drop(columns=['id','target'])
y_train = data_train_prepared['target']

X_val = data_val_prepared.drop(columns=['id','target'])
y_val = data_val_prepared['target']

#### Обучаем логрег

In [ ]:
# обучаем логрег

model_lr = LogisticRegression(
    multi_class='multinomial',  
    solver='lbfgs',              
    C=1.0,                       
    penalty='l2',                
    max_iter=1000,               
    class_weight='balanced',     
    random_state=RANDOM_STATE,   
    n_jobs=-1                    
)
model_lr.fit(X_train, y_train)
'''
при разных уровнях max_iter
max_iter=500: ROC-AUC=0.7962
max_iter=1000: ROC-AUC=0.8087
max_iter=1500: ROC-AUC=0.8099
max_iter=2000: ROC-AUC=0.8118
max_iter=3000: ROC-AUC=0.8126
'''

y_pred = model_lr.predict(X_val)

# считаем метрики для прогнозов
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='weighted')
recall = recall_score(y_val, y_pred, average='macro')
f1 = f1_score(y_val, y_pred, average='weighted')

# ROC-AUC — требует вероятности, не классы
y_proba = model_lr.predict_proba(X_val)
roc_auc = roc_auc_score(y_val, y_proba, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_val, y_pred))

In [ ]:
# попробуем подобрать оптимальные пороги для всех классов
classes = np.unique(y_val)
y_val_bin = label_binarize(y_val, classes=classes)

thresholds = np.linspace(0, 1, 101)
best_thresholds = []
for i, c in enumerate(classes):
    f1_scores = [f1_score(y_val_bin[:, i], (y_proba[:, i] > t).astype(int)) for t in thresholds]
    best_t = thresholds[np.argmax(f1_scores)]
    best_thresholds.append(best_t)
    print(f"Класс {c}: оптимальный порог {best_t:.3f}")

In [ ]:
# применяем индивидуальные пороги для расстановки классов
y_pred_custom = []
for probas in y_proba:
    # пробегаем по всем классам, применяем пороги
    adjusted = (probas >= best_thresholds).astype(int)
    if adjusted.sum() == 0:
        # если ни один класс не "прошёл", берём тот, который с max вероятностью
        adjusted[np.argmax(probas)] = 1
    y_pred_custom.append(np.argmax(adjusted) + 1)

y_pred_custom = np.array(y_pred_custom)

In [ ]:
# теперь считаем метрики для прогнозов с оптимальными порогами
accuracy = accuracy_score(y_val, y_pred_custom)
precision = precision_score(y_val, y_pred_custom, average='weighted')
recall = recall_score(y_val, y_pred_custom, average='macro')
f1 = f1_score(y_val, y_pred_custom, average='weighted')

y_proba = model_lr.predict_proba(X_val)
roc_auc = roc_auc_score(y_val, y_proba, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_val, y_pred))

По итогу видим, что **Accuracy** выросла (с 0.40 до 0.59 против), в ущерб остальным метрикам **Precision** сильно упала (с 0.79 до 0.34), **Recall** упала (с 0.44 до 0.11), **F1-score** ухудшилось (с 0.49 до 0.43) - изменение порога делает распределение некорректным (классы становятся неравноправными), алгоритм начинает отдавать предпочтение тем классам, где порог легче пройти, видим просадку по recall почти до нуля.

In [ ]:
# попробуем еще "поиграть" с данными, чтобы избавиться от дисбаланса классов
# сначала погенерируем мелкие классы и уберем "шумные" примеры классов
smote_enn = SMOTEENN(random_state=RANDOM_STATE)
X_train_senn, y_train_senn = smote_enn.fit_resample(X_train, y_train)

print(y_train_senn.value_counts(normalize=True))

In [ ]:
# и просто погенерируем мелкие классы

smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(y_train_res.value_counts(normalize=True))

In [ ]:
# обучаем логрег_1
model_lr_1 = LogisticRegression(
    multi_class='multinomial',   
    solver='lbfgs',             
    C=1.0,                       
    penalty='l2',                
    max_iter=1000,               
    class_weight='balanced',     
    random_state=RANDOM_STATE,   
    n_jobs=-1                    
)

model_lr_1.fit(X_train_senn, y_train_senn)

y_pred = model_lr_1.predict(X_val)

# считаем метрики для прогнозов
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='weighted')
recall = recall_score(y_val, y_pred, average='macro')
f1 = f1_score(y_val, y_pred, average='weighted')

y_proba = model_lr_1.predict_proba(X_val)
roc_auc = roc_auc_score(y_val, y_proba, multi_class='ovr')


print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_val, y_pred))

In [ ]:
# или просто уменьшим мажоритарные примеры классов, средние и малые оставим как есть
counts = y_train.value_counts()

sampling_strategy = {
    1: int(counts[1] * 0.1),  
    2: int(counts[2] * 0.3),   
    3: counts[3],              
    4: counts[4],
    5: counts[5],
    6: counts[6],
    7: counts[7],
    8: counts[8],
    9: counts[9],
}

rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=RANDOM_STATE)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

print(y_train_rus.value_counts(normalize=True).sort_index())

In [ ]:
# обучаем логрег_2
model_lr_2 = LogisticRegression(
    multi_class='multinomial',   
    solver='lbfgs',              
    C=1.0,                       
    penalty='l2',                
    max_iter=1000,               
    class_weight='balanced',     
    random_state=RANDOM_STATE,   
    n_jobs=-1                    
)

model_lr_2.fit(X_train_rus, y_train_rus)

y_pred = model_lr_2.predict(X_val)

# считаем метрики для прогнозов

accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(y_val, y_pred, average='weighted')
recall = recall_score(y_val, y_pred, average='macro')
f1 = f1_score(y_val, y_pred, average='weighted')

y_proba = model_lr_2.predict_proba(X_val)
roc_auc = roc_auc_score(y_val, y_proba, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_val, y_pred))

На базовой модели логистической регресии получили ожидаемо слабые метрики, которые скорее всего связаны с нелинейными зависимостями (текстовых-то эмбеддингах), при этом изменение баланса данных с допгенерацией признаков повлияло незначительно +2-3% на изменение метрик, а скоращение мажоритарных классов - ухудшило метрику.

#### Обучаем randomforest

In [ ]:
if os.path.exists("model_rf.pkl"):
    best_model_rf = RandomForestClassifier()
    best_model_rf = joblib.load("model_rf.pkl")
    print("Параметры модели:", best_model_rf.get_params())
else:

    # создаем модель
    model_rf = RandomForestClassifier( 
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=2
    )


    # сетка гиперпараметров
    param_grid = [
        # RandomForestClassifier
        {
            "n_estimators": [1000],
            "max_depth": [9],
            "min_samples_split": [2],
            "min_samples_leaf": [1],
            "class_weight": ["balanced"]
        }
    ]

    # полный перебор гиперпараметров с помощью GridSearchCV
    grid_search_rf = GridSearchCV(
        model_rf,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=5),
        scoring="f1_weighted",
        # error_score=np.nan
    )

    # обучение модели
    grid_search_rf.fit(X_train, y_train)

    # выбираем лучшаую модель и сохраняем 
    best_model_rf = grid_search_rf.best_estimator_    
    joblib.dump(best_model_rf, "model_rf.pkl")
    print('Параметры лучшей модели: ', best_model_rf.get_params())

# предсказания
y_pred_rf = best_model_rf.predict(X_val)

# метрики
accuracy = accuracy_score(y_val, y_pred_rf)
precision = precision_score(y_val, y_pred_rf, average='weighted')
recall = recall_score(y_val, y_pred_rf, average='macro')
f1 = f1_score(y_val, y_pred_rf, average='weighted')

y_proba_rf = best_model_rf.predict_proba(X_val)
roc_auc = roc_auc_score(y_val, y_proba_rf, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_val, y_pred_rf))

In [ ]:
# посмотрим важности признаков
importances = best_model_rf.feature_importances_

feat_imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(20))

In [ ]:
# попробуем обучить randomforest без последних 4 признаков
'''
# выбрасываем признаки
X_train_drop4 = X_train.drop(columns = ['month','day_of_week_sin','day_of_week','is_weekend'])
X_val_drop4 = X_val.drop(columns = ['month','day_of_week_sin','day_of_week','is_weekend'])

# создаем модель
model_rf_1 = RandomForestClassifier(
    n_estimators=500,         
    max_depth=9,           
    min_samples_split=2,      
    min_samples_leaf=1,       
    class_weight='balanced',  
    random_state=RANDOM_STATE,
    n_jobs=-1                 
)

# обучаем
model_rf_1.fit(X_train_drop4, y_train)

# предсказания
y_pred_rf_1 = model_rf_1.predict(X_val_drop4)

# метрики
accuracy = accuracy_score(y_val, y_pred_rf_1)
precision = precision_score(y_val, y_pred_rf_1, average='weighted')
recall = recall_score(y_val, y_pred_rf_1, average='macro')
f1 = f1_score(y_val, y_pred_rf_1, average='weighted')

y_proba_rf_1 = model_rf_1.predict_proba(X_val_drop4)
roc_auc = roc_auc_score(y_val, y_proba_rf_1, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_val, y_pred_rf_1))
'''

In [ ]:
# посмотрим важности признаков
'''
importances = model_rf_1.feature_importances_

feat_imp = pd.DataFrame({
    'feature': X_train_drop4.columns,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(20))
'''

Отбрасывание малозначительных признаков никак не повлияло на результативность классификации.

#### Обучаем catboost

In [27]:
# если есть обученная модель на сервере, загружаем ее, иначе запускаем свое обучение

if os.path.exists("model_cb.cbm"):
    best_model_cb = CatBoostClassifier()
    best_model_cb.load_model("model_cb.cbm")
    print('Параметры загруженной модели: ', best_model_cb.get_params())
else:

    if utils.get_gpu_device_count() > 0:
        task_type = "GPU"
    else:
        task_type = "CPU"
    
    #cat_features = ["day_of_week", "month", "is_weekend"]
    cat_features = []
        
    # создаем модель
    model_cb = CatBoostClassifier(
                            #silent=True,
                            random_seed=RANDOM_STATE, 
                            cat_features=cat_features,
                            task_type=task_type,
                            thread_count=-1,
                            verbose = 100
    )

    # сетка гиперпараметров
    param_grid = [
        # CatBoostClassifier
        {
            "iterations": [1000],
            "depth": [6],
            "learning_rate": [0.05],
            "l2_leaf_reg": [9],
            "loss_function": ["MultiClassOneVsAll"], #MultiClassOneVsAll, MultiClass
            "max_bin": [256],
            "random_strength": [5],
            "early_stopping_rounds": [10],
            "min_data_in_leaf":[1],
            "auto_class_weights":['SqrtBalanced']
        }
    ]

    # полный перебор гиперпараметров с помощью GridSearchCV
    grid_search_cb = GridSearchCV(
        model_cb,
        param_grid=param_grid,
        cv=StratifiedKFold(n_splits=2),
        scoring="f1_weighted",
        n_jobs=1,
        refit=True
        # error_score=np.nan
    )

    # обучение модели
    grid_search_cb.fit(X_train, y_train,
                       eval_set=(X_val, y_val), 
                       use_best_model=True)

    # выбираем лучшаую модель и сохраняем 
    best_model_cb = grid_search_cb.best_estimator_    
    best_model_cb.save_model("model_cb.cbm")
    print('Параметры лучшей модели: ', best_model_cb.get_params())

# предсказания
y_pred_cb = best_model_cb.predict(X_val)

# метрики
accuracy = accuracy_score(y_val, y_pred_cb)
precision = precision_score(y_val, y_pred_cb, average='weighted')
recall = recall_score(y_val, y_pred_cb, average='macro')
f1 = f1_score(y_val, y_pred_cb, average='weighted')

y_proba_cb = best_model_cb.predict_proba(X_val)
y_proba_cb = y_proba_cb / y_proba_cb.sum(axis=1, keepdims=True) # при обучении модели с функцией MultiClassOneVsAll
roc_auc = roc_auc_score(y_val, y_proba_cb, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_val, y_pred_cb))

0:	learn: 0.6522127	test: 0.6517168	best: 0.6517168 (0)	total: 25.7ms	remaining: 25.7s
100:	learn: 0.0791747	test: 0.0659671	best: 0.0659671 (100)	total: 1.42s	remaining: 12.6s
200:	learn: 0.0532973	test: 0.0412271	best: 0.0412271 (200)	total: 2.6s	remaining: 10.4s
300:	learn: 0.0431717	test: 0.0324222	best: 0.0324222 (300)	total: 3.81s	remaining: 8.85s
400:	learn: 0.0375633	test: 0.0280202	best: 0.0280202 (400)	total: 5s	remaining: 7.46s
500:	learn: 0.0334930	test: 0.0251368	best: 0.0251368 (500)	total: 6.17s	remaining: 6.15s
600:	learn: 0.0306715	test: 0.0232281	best: 0.0232281 (600)	total: 7.35s	remaining: 4.88s
700:	learn: 0.0281921	test: 0.0217205	best: 0.0217205 (700)	total: 8.54s	remaining: 3.64s
800:	learn: 0.0263419	test: 0.0205525	best: 0.0205525 (800)	total: 9.71s	remaining: 2.41s
900:	learn: 0.0246909	test: 0.0196386	best: 0.0196386 (900)	total: 10.9s	remaining: 1.2s
999:	learn: 0.0233409	test: 0.0189188	best: 0.0189188 (999)	total: 12.1s	remaining: 0us
bestTest = 0.0189187

In [11]:
# проверяем важности
importances = best_model_cb.get_feature_importance()

feat_imp = pd.DataFrame({
    'feature': best_model_cb.feature_names_,
    'importance': importances
}).sort_values(by='importance', ascending=False)

print(feat_imp.head(20))

                      feature  importance
7              articles_pca_3   15.554475
5              articles_pca_1   14.845512
6              articles_pca_2   11.910314
13       counterparties_pca_1    8.492066
15       counterparties_pca_3    8.222650
4         articles__name_mean    6.934604
3               purpose_pca_3    5.527749
1               purpose_pca_1    4.992319
2               purpose_pca_2    4.442037
14       counterparties_pca_2    4.385346
11             projects_pca_3    4.100752
10             projects_pca_2    3.338315
9              projects_pca_1    3.016482
12  counterparties__name_mean    2.165234
0                purpose_mean    1.573374
8         projects__name_mean    0.498772


### Тестирование моделей

In [12]:
# загружаем тестовые датасеты
data_test = pd.read_csv('data_test.csv')
data_test_corr_part = pd.read_csv('data_test_corr_part.csv')
data_test_corr_full = pd.read_csv('data_test_corr_full.csv')

# сверим их
display(data_test.head(),
        data_test_corr_part.head(),
        data_test_corr_full.head())

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,universal_category
0,696326,2368,42969,2024-12-02,2131157.00,Перечисление средств по соглашению №2140 о предоставлении гранта победителю конкурса на предоставление грантов Раиса...,39920,incoming,3906,0,0,-1,0,2368.0,942.0,39920.0,942.0,41881.0,Гранты,3906.0,942.0,0.0,Донорство в Татарстане,NaN,NaN,NaN,NaN,NaN,NaN,NaN,гранты субсидии конкурсы
1,750785,725,51,2025-02-27,1500.00,ЗА 27/02/2025;СУМ:1500.00;КОМ:0.00;СЛЮСАРЕНКО ВИКТОРИЯ АНАТОЛЬЕВНА;Попечительский взнос котёнок Тужур,29040,incoming,1831,0,0,1500,6228,725.0,114.0,29040.0,114.0,28282.0,Попечительские,1831.0,114.0,1829.0,Кошкин дом,6228.0,114.0,6226.0,... массовые,1500.0,114.0,NaN,членские и учредительские взносы
2,671791,1027,46,2025-01-21,0.22,"Перевод с карты *6947, Благотворительное пожертвование на уставную деятельность. НДС не облагается",33352,incoming,2311,0,0,7148,7389,1027.0,804.0,33352.0,804.0,33347.0,Юр.лица,2311.0,804.0,0.0,Травли NET,7389.0,804.0,7387.0,?_нужно_проставить_донора,NaN,NaN,NaN,пожертвования от юридических лиц (напрямую)
3,991063,1837,70267,2025-06-20,80143.68,"ОПЛАТА ПО ДОГОВОРУ НХТК.8190 (ДС НХТК.8190-2), СЧЕТ №21 ОТ 02.06.2025 (КВОТИРУЕМЫЕ РАБОЧИЕ МЕСТА). СУММА 80143-68 РУ...",36939,incoming,2995,0,1849,7204,0,1837.0,875.0,36939.0,875.0,36936.0,Квота - НХТК,2995.0,875.0,2991.0,Квота - НХТК,NaN,NaN,NaN,NaN,7204.0,875.0,NaN,продажа услуг
4,847821,1448,1482,2025-02-03,310750.00,БЛАГОТВОРИТЕЛЬНАЯ ПОМОЩЬ ПО СОГЛАШЕНИЮ №БС2025-005 ОТ 16.01.2025 СУММА 310750-00 БЕЗ НАЛОГА (НДС),34599,incoming,2476,0,0,-1,10258,1448.0,822.0,34599.0,822.0,34598.0,Гранты,2476.0,822.0,0.0,Э_надо_разобрать,10258.0,822.0,11239.0,РЕК 2025,NaN,NaN,NaN,гранты субсидии конкурсы


,index,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,universal_category
0,0,696326,2368,42969,2024-12-02,2131157.00,Перечисление средств по соглашению №2140 о предоставлении гранта победителю конкурса на предоставление грантов Раиса...,39920,incoming,3906,0,0,-1,0,2368.0,942.0,39920.0,942.0,41881.0,Грантик,3906.0,942.0,0.0,Донорство в Татарстане,NaN,NaN,NaN,NaN,NaN,NaN,NaN,гранты субсидии конкурсы
1,180,969238,3735,111749,2025-02-05,1174227.83,субсидия за февраль 2025,44999,incoming,4279,0,1951,-1,0,3735.0,1004.0,44999.0,1004.0,44997.0,Социальная дем.,4279.0,1004.0,4277.0,Присмотр и уход,NaN,NaN,NaN,NaN,NaN,NaN,NaN,гранты субсидии конкурсы
2,179,113599,160,15591,2022-04-26,2220593.00,(93810061540906600633246=2220593;02452000150)Субс-я на фин. обеспеч-е затрат соц. ориентир. некомер. организ-и в 202...,7859,incoming,735,0,0,-1,5651,160.0,176.0,7859.0,176.0,7857.0,Гран,735.0,176.0,514.0,Станем сильнее вместе,5651.0,176.0,1809.0,Грант губернатора ЛО-2022-2023,NaN,NaN,NaN,гранты субсидии конкурсы
3,178,826672,2980,96992,2024-03-18,966895.36,Перечисление средств для открытия депозита по депозитной сделке № UNV-0000001249512 от 18.03.2024 согласно заявлению...,27271,incoming,3256,0,0,-1,8065,2980.0,731.0,27271.0,731.0,27269.0,гранти,3256.0,731.0,3253.0,игрец,8065.0,731.0,5876.0,средние,NaN,NaN,NaN,гранты субсидии конкурсы
4,866,1035131,3568,18504,2025-09-02,1870128.00,Перечисление пожертвования по договору № Т-Гранты/НКО-25-000912 от 01.09.2025.НДС не облагается.,44027,incoming,4165,0,0,-1,12287,3568.0,1010.0,44027.0,1010.0,44026.0,Гран,4165.0,1010.0,4163.0,Административные расходы,12287.0,1010.0,11005.0,Девелопмент Групп,NaN,NaN,NaN,гранты субсидии конкурсы


,index,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,universal_category
0,0,696326,2368,42969,2024-12-02,2131157.00,Перечисление средств по соглашению №2140 о предоставлении гранта победителю конкурса на предоставление грантов Раиса...,39920,incoming,3906,0,0,-1,0,2368.0,942.0,39920.0,942.0,41881.0,Грнт,3906.0,942.0,0.0,Татарстане вДонорство,NaN,NaN,NaN,NaN,NaN,NaN,NaN,гранты субсидии конкурсы
1,180,969238,3735,111749,2025-02-05,1174227.83,субсидия за февраль 2025,44999,incoming,4279,0,1951,-1,0,3735.0,1004.0,44999.0,1004.0,44997.0,Сц.дм.,4279.0,1004.0,4277.0,ухоa и Присмотр,NaN,NaN,NaN,NaN,NaN,NaN,NaN,гранты субсидии конкурсы
2,179,113599,160,15591,2022-04-26,2220593.00,(93810061540906600633246=2220593;02452000150)Субс-я на фин. обеспеч-е затрат соц. ориентир. некомер. организ-и в 202...,7859,incoming,735,0,0,-1,5651,160.0,176.0,7859.0,176.0,7857.0,Грнат,735.0,176.0,514.0,вместеьнее Станем,5651.0,176.0,1809.0,губернатора Грант ЛО-2022-2023,NaN,NaN,NaN,гранты субсидии конкурсы
3,178,826672,2980,96992,2024-03-18,966895.36,Перечисление средств для открытия депозита по депозитной сделке № UNV-0000001249512 от 18.03.2024 согласно заявлению...,27271,incoming,3256,0,0,-1,8065,2980.0,731.0,27271.0,731.0,27269.0,Грнат,3256.0,731.0,3253.0,грy,8065.0,731.0,5876.0,средние,NaN,NaN,NaN,гранты субсидии конкурсы
4,866,1035131,3568,18504,2025-09-02,1870128.00,Перечисление пожертвования по договору № Т-Гранты/НКО-25-000912 от 01.09.2025.НДС не облагается.,44027,incoming,4165,0,0,-1,12287,3568.0,1010.0,44027.0,1010.0,44026.0,т,4165.0,1010.0,4163.0,рсхд дмнсартвн,12287.0,1010.0,11005.0,ДевелопментГрупп,NaN,NaN,NaN,гранты субсидии конкурсы


In [13]:
# загружаем скейлеры для эмбеддингов и преобразовываем тестовые наборы
scaler_emb_path = 'scaler.pkl'
ipca_emb_path = 'ipca.pkl'
scaler_art_emb_path = 'scaler_art.pkl'
ipca_art_emb_path = 'ipca_art.pkl'
scaler_pro_emb_path = 'scaler_pro.pkl'
ipca_pro_emb_path = 'ipca_pro.pkl'
scaler_cou_emb_path = 'scaler_cou.pkl'
ipca_cou_emb_path = 'ipca_cou.pkl'

scaler = joblib.load(scaler_emb_path)
ipca = joblib.load(ipca_emb_path)
scaler_art = joblib.load(scaler_art_emb_path)
ipca_art = joblib.load(ipca_art_emb_path)
scaler_pro = joblib.load(scaler_pro_emb_path)
ipca_pro = joblib.load(ipca_pro_emb_path)
scaler_cou = joblib.load(scaler_cou_emb_path)
ipca_cou = joblib.load(ipca_cou_emb_path)


data_test_prepared, _,_,_,_,_,_,_,_ = prepare_data(data_test, is_train=False, scaler=scaler, ipca=ipca,
                                                                   scaler_art=scaler_art, ipca_art=ipca_art,
                                                                   scaler_pro=scaler_pro, ipca_pro=ipca_pro,
                                                                   scaler_cou=scaler_cou, ipca_cou=ipca_cou)

data_test_corr_part_prepared, _,_,_,_,_,_,_,_ = prepare_data(data_test_corr_part, is_train=False, scaler=scaler, ipca=ipca,
                                                                                       scaler_art=scaler_art, ipca_art=ipca_art,
                                                                                       scaler_pro=scaler_pro, ipca_pro=ipca_pro,
                                                                                       scaler_cou=scaler_cou, ipca_cou=ipca_cou)

data_test_corr_full_prepared, _,_,_,_,_,_,_,_ = prepare_data(data_test_corr_full, is_train=False, scaler=scaler, ipca=ipca,
                                                                                       scaler_art=scaler_art, ipca_art=ipca_art,
                                                                                       scaler_pro=scaler_pro, ipca_pro=ipca_pro,
                                                                                       scaler_cou=scaler_cou, ipca_cou=ipca_cou)

data_test_prepared.to_csv('data_test_prepared.csv',index=False)
data_test_corr_part_prepared.to_csv('data_test_corr_part_prepared.csv',index=False)
data_test_corr_full_prepared.to_csv('data_test_corr_full_prepared.csv',index=False)


🔍 Запуск предобработки для 900 текстов...


Очистка: 100%|██████████| 900/900 [00:00<00:00, 179380.04it/s]

🔍 Загрузка spaCy модели: 'ru_core_news_sm'.


🔍 Лемматизация будет выполнена в 3 потоках.


Лемматизация (CPU): 100%|██████████| 900/900 [00:03<00:00, 275.50it/s]


...

Лемматизация (CPU): 100%|██████████| 900/900 [00:01<00:00, 778.38it/s] 


🔍 Предобработка завершена. Обработано 900 текстов.
🔍 Начало генерации эмбеддингов для 900 текстов на устройстве 'cuda'.


Генерация эмбеддингов: 100%|██████████| 15/15 [00:00<00:00, 21.80it/s]


🔍 Генерация эмбеддингов завершена


Применяем PCA к counterparties embeddings: 100%|██████████| 1/1 [00:00<00:00, 90.35it/s]


In [14]:
# разделим признаки
X_data_test_prepared = data_test_prepared.drop(columns=['id','target'])
y_data_test_prepared = data_test_prepared['target']

X_data_test_corr_part_prepared = data_test_corr_part_prepared.drop(columns=['index','id','target'])
y_data_test_corr_part_prepared = data_test_corr_part_prepared['target']

X_data_test_corr_full_prepared = data_test_corr_full_prepared.drop(columns=['index','id','target'])
y_data_test_corr_full_prepared = data_test_corr_full_prepared['target']


#### Тестируем randomforest и catboost на исходном тестовом наборе без корректировок

In [ ]:
# предсказания rf
y_pred_rf = best_model_rf.predict(X_data_test_prepared)

# метрики
accuracy = accuracy_score(y_data_test_prepared, y_pred_rf)
precision = precision_score(y_data_test_prepared, y_pred_rf, average='weighted')
recall = recall_score(y_data_test_prepared, y_pred_rf, average='macro')
f1 = f1_score(y_data_test_prepared, y_pred_rf, average='weighted')

y_proba_rf = best_model_rf.predict_proba(X_data_test_prepared)
roc_auc = roc_auc_score(y_data_test_prepared, y_proba_rf, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_data_test_prepared, y_pred_rf))

In [15]:
# предсказания catboost
y_pred_cb = best_model_cb.predict(X_data_test_prepared)

# метрики
accuracy = accuracy_score(y_data_test_prepared, y_pred_cb)
precision = precision_score(y_data_test_prepared, y_pred_cb, average='weighted')
recall = recall_score(y_data_test_prepared, y_pred_cb, average='macro')
f1 = f1_score(y_data_test_prepared, y_pred_cb, average='weighted')

y_proba_cb = best_model_cb.predict_proba(X_data_test_prepared)
y_proba_cb = y_proba_cb / y_proba_cb.sum(axis=1, keepdims=True) # при обучении модели с функцией MultiClassOneVsAll
roc_auc = roc_auc_score(y_data_test_prepared, y_proba_cb, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_data_test_prepared, y_pred_cb))

Accuracy: 0.9677777777777777
Precision (weighted): 0.9702780273754373
Recall (macro): 0.9678746922311279
F1-score (weighted): 0.9673746220284078
ROC-AUC: 0.9998253931124892

Отчет по классам:
              precision    recall  f1-score   support

           1       0.88      1.00      0.94        99
           2       0.93      1.00      0.97       100
           3       0.99      0.96      0.97       101
           4       1.00      1.00      1.00       100
           5       0.95      0.99      0.97       105
           6       0.98      1.00      0.99        95
           7       1.00      0.97      0.98       100
           8       1.00      0.97      0.98       100
           9       1.00      0.82      0.90       100

    accuracy                           0.97       900
   macro avg       0.97      0.97      0.97       900
weighted avg       0.97      0.97      0.97       900



#### Тестируем randomforest и catboost на частично скорректированном тестовом наборе (аналог для тестов классификации c LLM)

In [ ]:
# предсказания rf
y_pred_rf = best_model_rf.predict(X_data_test_corr_part_prepared)

# метрики
accuracy = accuracy_score(y_data_test_corr_part_prepared, y_pred_rf)
precision = precision_score(y_data_test_corr_part_prepared, y_pred_rf, average='weighted')
recall = recall_score(y_data_test_corr_part_prepared, y_pred_rf, average='macro')
f1 = f1_score(y_data_test_corr_part_prepared, y_pred_rf, average='weighted')

y_proba_rf = best_model_rf.predict_proba(X_data_test_corr_part_prepared)
roc_auc = roc_auc_score(y_data_test_corr_part_prepared, y_proba_rf, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_data_test_corr_part_prepared, y_pred_rf))

In [23]:
# предсказания catboost
y_pred_cb = best_model_cb.predict(X_data_test_corr_part_prepared)

# метрики
accuracy = accuracy_score(y_data_test_corr_part_prepared, y_pred_cb)
precision = precision_score(y_data_test_corr_part_prepared, y_pred_cb, average='weighted')
recall = recall_score(y_data_test_corr_part_prepared, y_pred_cb, average='macro')
f1 = f1_score(y_data_test_corr_part_prepared, y_pred_cb, average='weighted')

y_proba_cb = best_model_cb.predict_proba(X_data_test_corr_part_prepared)
y_proba_cb = y_proba_cb / y_proba_cb.sum(axis=1, keepdims=True) # при обучении модели с функцией MultiClassOneVsAll
roc_auc = roc_auc_score(y_data_test_corr_part_prepared, y_proba_cb, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_data_test_corr_part_prepared, y_pred_cb))

Accuracy: 0.8233333333333334
Precision (weighted): 0.8326488487195643
Recall (macro): 0.8229361139956936
F1-score (weighted): 0.8219794242721216
ROC-AUC: 0.9812404084400804

Отчет по классам:
              precision    recall  f1-score   support

           1       0.79      0.87      0.83        99
           2       0.89      0.98      0.93       100
           3       0.83      0.95      0.88       101
           4       0.74      0.64      0.69       100
           5       0.68      0.88      0.76       105
           6       0.84      0.82      0.83        95
           7       0.86      0.83      0.85       100
           8       0.94      0.72      0.81       100
           9       0.94      0.72      0.81       100

    accuracy                           0.82       900
   macro avg       0.83      0.82      0.82       900
weighted avg       0.83      0.82      0.82       900



#### Тестируем randomforest и catboost на значительно скорректированном тестовом наборе

In [ ]:
# предсказания rf
y_pred_rf = best_model_rf.predict(X_data_test_corr_full_prepared)

# метрики
accuracy = accuracy_score(y_data_test_corr_full_prepared, y_pred_rf)
precision = precision_score(y_data_test_corr_full_prepared, y_pred_rf, average='weighted')
recall = recall_score(y_data_test_corr_full_prepared, y_pred_rf, average='macro')
f1 = f1_score(y_data_test_corr_full_prepared, y_pred_rf, average='weighted')

y_proba_rf = best_model_rf.predict_proba(X_data_test_corr_full_prepared)
roc_auc = roc_auc_score(y_data_test_corr_full_prepared, y_proba_rf, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_data_test_corr_full_prepared, y_pred_rf))

In [28]:
# предсказания catboost
y_pred_cb = best_model_cb.predict(X_data_test_corr_full_prepared)

# метрики
accuracy = accuracy_score(y_data_test_corr_full_prepared, y_pred_cb)
precision = precision_score(y_data_test_corr_full_prepared, y_pred_cb, average='weighted')
recall = recall_score(y_data_test_corr_full_prepared, y_pred_cb, average='macro')
f1 = f1_score(y_data_test_corr_full_prepared, y_pred_cb, average='weighted')

y_proba_cb = best_model_cb.predict_proba(X_data_test_corr_full_prepared)
y_proba_cb = y_proba_cb / y_proba_cb.sum(axis=1, keepdims=True) # при обучении модели с функцией MultiClassOneVsAll
roc_auc = roc_auc_score(y_data_test_corr_full_prepared, y_proba_cb, multi_class='ovr')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)
print("ROC-AUC:", roc_auc)

print("\nОтчет по классам:")
print(classification_report(y_data_test_corr_full_prepared, y_pred_cb))

Accuracy: 0.5688888888888889
Precision (weighted): 0.6740891227138783
Recall (macro): 0.5695302136729965
F1-score (weighted): 0.5569222794461689
ROC-AUC: 0.9173783979079649

Отчет по классам:
              precision    recall  f1-score   support

           1       0.32      0.88      0.47        99
           2       0.76      0.96      0.85       100
           3       0.58      0.73      0.65       101
           4       0.70      0.68      0.69       100
           5       0.59      0.42      0.49       105
           6       0.49      0.51      0.50        95
           7       0.85      0.47      0.61       100
           8       0.83      0.25      0.38       100
           9       0.92      0.23      0.37       100

    accuracy                           0.57       900
   macro avg       0.67      0.57      0.56       900
weighted avg       0.67      0.57      0.56       900

